# Manga Diffusion — LoRA Training

Trains a LoRA on Manga109-s panels using kohya_ss sd-scripts.

**Before running:**
1. Runtime → Change runtime type → GPU (T4 or A100)
2. Set CONDITION and WANDB_API_KEY below
3. Upload your `data/train_set/` folder to Google Drive at the path in DRIVE_TRAIN_DATA

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not detected! Go to Runtime → Change runtime type → T4 GPU and re-run.")

gpu = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU : {gpu}")
print(f"VRAM: {mem:.1f} GB")
print("CUDA ready.")

In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────────
CONDITION       = "blip2"   # "blip2" or "wd14"
WANDB_API_KEY   = ""        # paste your W&B key, or leave blank to disable
DRIVE_TRAIN_DATA = "/content/drive/MyDrive/manga-diffusion/data/train_set"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/manga-diffusion/training/outputs"
# ──────────────────────────────────────────────────────────────────────────────

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted.")

## 2. Install dependencies

In [ ]:
%%bash
pip install -q accelerate transformers diffusers peft bitsandbytes xformers wandb toml

In [ ]:
%%bash
if [ ! -d "/content/kohya_ss" ]; then
    git clone -q https://github.com/kohya-ss/sd-scripts /content/kohya_ss
    cd /content/kohya_ss && pip install -q -r requirements.txt
    echo "kohya_ss installed."
else
    echo "kohya_ss already present."
fi

## 3. Set up W&B

In [ ]:
import os
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    import wandb; wandb.login(key=WANDB_API_KEY)
    print("W&B logged in.")
else:
    os.environ["WANDB_MODE"] = "disabled"
    print("W&B disabled — logs saved locally only.")

## 4. Verify training data

In [ ]:
import os, glob
train_dir = os.path.join(DRIVE_TRAIN_DATA, CONDITION, "10_gknoir")
images   = glob.glob(os.path.join(train_dir, "*.png"))
captions = glob.glob(os.path.join(train_dir, "*.txt"))
print(f"Condition : {CONDITION}")
print(f"Train dir : {train_dir}")
print(f"Images    : {len(images)}")
print(f"Captions  : {len(captions)}")
assert len(images) > 0, f"No images found in {train_dir}"
assert len(images) == len(captions), "Image/caption count mismatch!"
with open(captions[0]) as f:
    print(f"\nSample caption: {f.read()[:200]}")

## 5. Write kohya config

In [ ]:
import toml, os
run_name   = f"{CONDITION}-rank32"
output_dir = os.path.join(DRIVE_OUTPUT_DIR, run_name)
os.makedirs(output_dir, exist_ok=True)
train_dir  = os.path.join(DRIVE_TRAIN_DATA, CONDITION, "10_gknoir")

config = {
    "general": {"enable_bucket": True, "shuffle_caption": True,
                "caption_extension": ".txt", "keep_tokens": 1},
    "datasets": [{"image_dir": train_dir, "caption_extension": ".txt",
                  "resolution": 768, "batch_size": 1, "num_repeats": 10}],
    "network":  {"network_module": "networks.lora",
                 "network_dim": 32, "network_alpha": 16},
    "optimizer":{"optimizer_type": "AdamW8bit", "learning_rate": 1e-4,
                 "lr_scheduler": "cosine_with_restarts", "lr_warmup_steps": 100},
    "training": {"max_train_steps": 2000, "save_every_n_steps": 500,
                 "mixed_precision": "fp16", "xformers": True, "seed": 42,
                 "output_dir": output_dir, "output_name": run_name},
    "logging":  {"log_with": "wandb" if WANDB_API_KEY else "tensorboard",
                 "wandb_run_name": run_name,
                 "logging_dir": os.path.join(output_dir, "logs")},
    "model":    {"pretrained_model_name_or_path":
                     "stabilityai/stable-diffusion-xl-base-1.0",
                 "vae": "madebyollin/sdxl-vae-fp16-fix"},
}

config_path = f"/content/{run_name}.toml"
with open(config_path, "w") as f:
    toml.dump(config, f)
print(f"Config written: {config_path}")
print(f"Output dir    : {output_dir}")

## 6. Run training

In [ ]:
import subprocess, sys
cmd = [sys.executable, "/content/kohya_ss/sdxl_train_network.py",
       f"--config_file={config_path}"]
print("Starting training ...")
print(" ".join(cmd))
result = subprocess.run(cmd, cwd="/content/kohya_ss")
print(f"\nFinished with exit code: {result.returncode}")

## 7. Verify saved checkpoints

In [ ]:
import glob, os
checkpoints = sorted(glob.glob(os.path.join(output_dir, "*.safetensors")))
print(f"Checkpoints in {output_dir}:")
for ckpt in checkpoints:
    print(f"  {os.path.basename(ckpt)}  ({os.path.getsize(ckpt)/1e6:.1f} MB)")
if not checkpoints:
    print("No checkpoints found — check training output above for errors.")

## 8. Quick inference test

In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline
from IPython.display import display

checkpoints = sorted(glob.glob(os.path.join(output_dir, "*.safetensors")))
if not checkpoints:
    print("No checkpoint to test.")
else:
    pipe = StableDiffusionXLPipeline.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0",
        torch_dtype=torch.float16, use_safetensors=True).to("cuda")
    pipe.load_lora_weights(checkpoints[-1])

    prompt = ("gknoir style, heavy ink linework, close-up panel, "
              "detective in rain, tense atmosphere, deep shadows")
    image = pipe(prompt=prompt, num_inference_steps=30, guidance_scale=7.5,
                 generator=torch.manual_seed(42)).images[0]
    display(image)
    out_path = os.path.join(output_dir, "test_generation.png")
    image.save(out_path)
    print(f"Test image saved to {out_path}")